In [4]:
!pip install ollama

In [16]:
# --- 0) Imports ---
import os
import json
import time
import re
from pathlib import Path

import ollama


# Find repo root
def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for p in [start] + list(start.parents):
        if (p / ".git").exists() or (p / "README.md").exists():
            return p
    raise RuntimeError(f"Repo root not found from start={start}")


repo_root = find_repo_root()
print("Repo root:", repo_root)

print("Ollama client ready.")


Repo root: /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer
Ollama client ready.


In [17]:
from pathlib import Path
from datetime import datetime, timezone
import re, json

# ------------------------------------------------------------------
# Dataset root
# ------------------------------------------------------------------
dataset_root = repo_root / "datasets" / "yt_factlink"
print("Dataset root:", dataset_root)

# ------------------------------------------------------------------
# Dataset ID (used in config metadata)
# ------------------------------------------------------------------

DATASET_ID = "yt_factlink"

# ------------------------------------------------------------------
# Choose evaluation source
# ------------------------------------------------------------------
USE_SPLIT = (
    True  # True = evaluate on test split, False = evaluate on full canonical file
)

SPLIT_NAME = "split_v1_seed42"  # used only if USE_SPLIT=True
# ------------------------------------------------------------------
# Choose provider + model to evaluate
# ------------------------------------------------------------------
PROVIDER = "ollama"
MODEL_NAME = "gpt-oss:120b"  # model you want to TEST
SUFFIX_TAG = "all_at_once"  # experiment name (prompt set)

# ------------------------------------------------------------------
# Dataset paths
# ------------------------------------------------------------------
if USE_SPLIT:
    SPLIT_DIR = dataset_root / "01_conversion" / "03_splits" / SPLIT_NAME
    EVAL_JSONL = SPLIT_DIR / "test.data.jsonl"

    assert SPLIT_DIR.exists(), f"Missing split folder: {SPLIT_DIR}"
    assert EVAL_JSONL.exists(), f"Missing test split: {EVAL_JSONL}"

    print("Evaluation mode: TEST SPLIT")
    print("Using split:", SPLIT_NAME)
else:
    EVAL_JSONL = (
        dataset_root
        / "01_conversion"
        / "02_outputs"
        / "manual_labels_386_v2.data.jsonl"
    )
    assert EVAL_JSONL.exists(), f"Canonical data JSONL missing: {EVAL_JSONL}"

    print("Evaluation mode: FULL CANONICAL (no split)")
    print("Using full dataset:", EVAL_JSONL)

print("Eval file:", EVAL_JSONL)


# ------------------------------------------------------------------
# Prompt paths (same idea as finetuning)
# ------------------------------------------------------------------
PROMPT_DIR = dataset_root / "02_prompts" / SUFFIX_TAG
SYSTEM_PATH = PROMPT_DIR / "system.txt"
USER_PATH = PROMPT_DIR / "user.txt"

assert SYSTEM_PATH.exists(), f"Missing system prompt: {SYSTEM_PATH}"
assert USER_PATH.exists(), f"Missing user prompt: {USER_PATH}"

print("Prompts loaded from:", PROMPT_DIR)

system_prompt = SYSTEM_PATH.read_text(encoding="utf-8")
user_template = USER_PATH.read_text(encoding="utf-8")


# ------------------------------------------------------------------
# Helper: slugify for safe folder names
# ------------------------------------------------------------------
def slugify(s: str) -> str:
    s = re.sub(r"[^a-zA-Z0-9._-]+", "-", s)
    s = re.sub(r"-{2,}", "-", s)
    return s.strip("-")


# ------------------------------------------------------------------
# Accuracy run folders (mirrors finetuning style)
# Notebook lives inside provider folder (e.g., openai/)
# Structure:
#   <NOTEBOOK_DIR>/<MODEL_NAME>/runs/<run_name>/
# ------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd()

MODEL_DIR = NOTEBOOK_DIR / slugify(MODEL_NAME)
MODEL_DIR.mkdir(exist_ok=True)
(MODEL_DIR / ".gitkeep").write_text("")

RUNS_ROOT = MODEL_DIR / "runs"
RUNS_ROOT.mkdir(exist_ok=True)
(RUNS_ROOT / ".gitkeep").write_text("")

timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H%M%SZ")

run_name = "_".join(
    [
        timestamp,
        slugify(PROVIDER),
        slugify(MODEL_NAME),
        slugify(SUFFIX_TAG),
        slugify(SPLIT_NAME),
    ]
)

RUN_DIR = RUNS_ROOT / run_name
RUN_DIR.mkdir(parents=True, exist_ok=False)

print("Notebook directory:", NOTEBOOK_DIR.resolve())
print("Model directory   :", MODEL_DIR.resolve())
print("Runs root         :", RUNS_ROOT.resolve())
print("Run directory     :", RUN_DIR.resolve())

# ------------------------------------------------------------------
# Create run subfolders early (so later cells can write into them)
# ------------------------------------------------------------------
INPUTS_DIR = RUN_DIR / "01_inputs"
OUTPUTS_DIR = RUN_DIR / "02_outputs"
SNAPSHOTS_DIR = RUN_DIR / "03_snapshots"

INPUTS_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)
SNAPSHOTS_DIR.mkdir(exist_ok=True)

(INPUTS_DIR / ".gitkeep").write_text("")
(OUTPUTS_DIR / ".gitkeep").write_text("")
(SNAPSHOTS_DIR / ".gitkeep").write_text("")

# Save prompt snapshots into inputs for reproducibility
(INPUTS_DIR / "system.txt").write_text(system_prompt, encoding="utf-8")
(INPUTS_DIR / "user.txt").write_text(user_template, encoding="utf-8")

# Save only the relative path of the eval dataset (no copying)
relative_eval_path = str(EVAL_JSONL.relative_to(repo_root))

(INPUTS_DIR / "eval_file.txt").write_text(relative_eval_path, encoding="utf-8")

print("Saved eval file reference:", relative_eval_path)

print("\n=== Configuration OK ===")
print("Provider   :", PROVIDER)
print("Model      :", MODEL_NAME)
print("Prompt tag :", SUFFIX_TAG)
print("Split      :", SPLIT_NAME)
print("Run dir    :", RUN_DIR)

Dataset root: /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink
Evaluation mode: TEST SPLIT
Using split: split_v1_seed42
Eval file: /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/01_conversion/03_splits/split_v1_seed42/test.data.jsonl
Prompts loaded from: /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/02_prompts/all_at_once
Notebook directory: /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_accuracy_testing/all_at_once/local
Model directory   : /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_accuracy_testing/all_at_once/local/gpt-oss-120b
Runs root         : /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_accuracy_testing/all_at_once/local/gpt-oss-120b/runs
Run directory     : /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_accuracy_testing/all_at_once/local/gpt-oss-120b/runs/2025-11-25T171301Z_ollama_gpt-oss

In [18]:
def read_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

test_rows = read_jsonl(EVAL_JSONL)
print("Test examples:", len(test_rows))

system_template = SYSTEM_PATH.read_text(encoding="utf-8")
user_template   = USER_PATH.read_text(encoding="utf-8")

# ---------------------------------------------------------------
# Copy prompt files into RUN_DIR/01_inputs for reproducibility
# ---------------------------------------------------------------
INPUTS_DIR = RUN_DIR / "01_inputs"
INPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Write frozen copies
(INPUTS_DIR / "system.txt").write_text(system_template, encoding="utf-8")
(INPUTS_DIR / "user.txt").write_text(user_template, encoding="utf-8")

print("Copied prompts to:", INPUTS_DIR.resolve())

Test examples: 77
Copied prompts to: /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_accuracy_testing/all_at_once/local/gpt-oss-120b/runs/2025-11-25T171301Z_ollama_gpt-oss-120b_all_at_once_split_v1_seed42/01_inputs


In [19]:
def build_user_message(row: dict) -> str:
    return (user_template
            .replace("[VIDEOTITLE]", row.get("VIDEOTITEL",""))
            .replace("[VIDEOCONTEXT]", row.get("VIDEOCONTEXT",""))
            .replace("[TARGETCOMMENT]", row.get("TARGETCOMMENT","")))

# preview one built message
print(build_user_message(test_rows[0])[:4000])


Input
Title:
史上首次！南非跟隨北京羞辱台灣，竟遭「晶片斷供」48小時後緊急求饒！北京當場看傻！

Video Context:
In September 2025, South Africa, bowing to Beijing's influence, demanded Taiwan downgrade its official diplomatic mission. Taiwan responded to this humiliation with an unprecedented and bold countermeasure: an export embargo on 47 critical high-tech products, including advanced semiconductors, to South Africa. This strategic move instantly exposed South Africa's deep structural dependence on Taiwan's technology, causing immediate domestic political fallout in Pretoria. The crisis forced South Africa to urgently seek dialogue with Taiwan just 48 hours after the embargo was announced.

This event signals a fundamental shift in Taiwan's international posture, transforming its "silicon shield" into a "silicon sword." Taiwan's assertiveness is underpinned by its booming capital markets and its dominant role in the global high-tech supply chain, especially in AI-related chips. The incident revealed a key strategic vulnera

In [20]:
# --- 3) Ollama 零樣本推論準備 ---
LABEL_KEYS = ["C1", "C2", "C3", "C4", "C6"]
OLLAMA_MODEL_NAME = MODEL_NAME
OLLAMA_SYSTEM_PROMPT = (
    system_prompt.strip()
    + "\n\n請嚴格以 JSON 物件回覆，格式為 {\"C1\":0/1, \"C2\":0/1, \"C3\":0/1, \"C4\":0/1, \"C6\":0/1}，不得加入額外說明。"
)


def parse_label_json(raw_text: str) -> dict | None:
    if not raw_text:
        return None
    match = re.search(r"\{.*\}", raw_text, flags=re.S)
    candidate = match.group(0) if match else raw_text
    candidate = candidate.replace("'", '"')
    try:
        data = json.loads(candidate)
    except json.JSONDecodeError:
        return None

    parsed = {}
    for key in LABEL_KEYS:
        value = data.get(key, 0)
        if isinstance(value, str):
            value = value.strip().lower()
            value = 1 if value in {"1", "true", "yes"} else 0
        parsed[key] = 1 if int(value) == 1 else 0
    return parsed


def call_ollama(messages: list[dict], max_retries: int = 3, sleep_sec: float = 0.2) -> str:
    last_err = None
    for _ in range(max_retries):
        try:
            response = ollama.chat(model=OLLAMA_MODEL_NAME, messages=messages)
            return response["message"]["content"]
        except Exception as exc:  # noqa: BLE001
            last_err = exc
            time.sleep(sleep_sec)
    raise RuntimeError(f"Ollama chat failed: {last_err}")



In [ ]:
# --- 4) 使用 Ollama 進行零樣本分類 ---
from IPython.display import clear_output

predictions = []
failures = 0

for i, row in enumerate(test_rows):
    user_msg = build_user_message(row)
    msgs = [
        {"role": "system", "content": OLLAMA_SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]

    raw_reply = ""
    parsed = None
    try:
        raw_reply = call_ollama(msgs)
        parsed = parse_label_json(raw_reply)
        if parsed is None:
            failures += 1
    except Exception as exc:  # noqa: BLE001
        failures += 1
        raw_reply = f"<ERROR: {exc}>"

    pred = parsed if parsed is not None else {k: 0 for k in LABEL_KEYS}
    predictions.append(
        {
            "pred": pred,
            "gold": row["labels"],
            "raw_output": raw_reply,
        }
    )

    clear_output(wait=True)
    print(f"[{i+1}/{len(test_rows)}] Failures: {failures}")
    print(row.get("TARGETCOMMENT"))
    print("GOLD:", row["labels"])
    print("PRED:", pred)

print("Done. Total failures:", failures)
predictions[0]



[71/77] Failures: 0
美好的佛教在台灣！感謝分享！
是法鼓山不是法古山，是聖嚴法師不是聖言法師
敬請更正，感恩！
GOLD: {'C1': 1, 'C2': 0, 'C3': 0, 'C4': 0, 'C6': 0}
PRED: {'C1': 0, 'C2': 0, 'C3': 0, 'C4': 0, 'C6': 0}


In [ ]:
def compute_metrics(predictions):
    # per-label confusion counts
    counts = {k: {"tp":0,"fp":0,"tn":0,"fn":0} for k in LABEL_KEYS}

    for ex in predictions:
        gold = ex["gold"]
        pred = ex["pred"]
        for k in LABEL_KEYS:
            g = int(gold.get(k,0))
            p = int(pred.get(k,0))
            if p==1 and g==1: counts[k]["tp"] += 1
            elif p==1 and g==0: counts[k]["fp"] += 1
            elif p==0 and g==0: counts[k]["tn"] += 1
            elif p==0 and g==1: counts[k]["fn"] += 1

    metrics = {}
    for k,c in counts.items():
        tp,fp,tn,fn = c["tp"],c["fp"],c["tn"],c["fn"]
        prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
        rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
        f1   = (2*prec*rec/(prec+rec)) if (prec+rec)>0 else 0.0
        acc  = (tp+tn)/(tp+tn+fp+fn) if (tp+tn+fp+fn)>0 else 0.0
        metrics[k] = {"precision":prec, "recall":rec, "f1":f1, "accuracy":acc, **c}

    # macro averages
    macro_f1 = sum(metrics[k]["f1"] for k in LABEL_KEYS)/len(LABEL_KEYS)
    macro_acc = sum(metrics[k]["accuracy"] for k in LABEL_KEYS)/len(LABEL_KEYS)

    return metrics, {"macro_f1": macro_f1, "macro_accuracy": macro_acc}

per_label, overall = compute_metrics(predictions)
per_label, overall


In [ ]:
import json
import shutil
from pathlib import Path

# -----------------------------
# Save run config into OUTPUTS_DIR
# -----------------------------
run_config = {
    "dataset_id": DATASET_ID,
    "experiment_type": "all_at_once",
    "provider": PROVIDER,
    "model_name": MODEL_NAME,
    "data_jsonl": str(EVAL_JSONL.relative_to(repo_root)),
    "prompt_dir": str(PROMPT_DIR.relative_to(repo_root)),
    "timestamp": timestamp,
}

(OUTPUTS_DIR / "run_config.json").write_text(
    json.dumps(run_config, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# -----------------------------
# Save predictions into OUTPUTS_DIR
# -----------------------------
with (OUTPUTS_DIR / "preds.jsonl").open("w", encoding="utf-8") as f:
    for ex in predictions:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

# -----------------------------
# Save full metrics into OUTPUTS_DIR
# -----------------------------
metrics_payload = {
    "failures": failures,
    "overall": overall,
    "per_label": per_label
}

(OUTPUTS_DIR / "metrics.json").write_text(
    json.dumps(metrics_payload, indent=2, ensure_ascii=False),
    encoding="utf-8"
)


# -----------------------------
# Snapshot THIS notebook into SNAPSHOTS_DIR
# -----------------------------
NOTEBOOK_FILENAME = "run_accuracy.ipynb"  # change if your notebook has a different name
NOTEBOOK_PATH = Path(NOTEBOOK_FILENAME)

if NOTEBOOK_PATH.exists():
    shutil.copy2(NOTEBOOK_PATH, SNAPSHOTS_DIR / NOTEBOOK_FILENAME)
    print("Notebook snapshot saved to:", (SNAPSHOTS_DIR / NOTEBOOK_FILENAME).resolve())
else:
    print(f"WARNING: {NOTEBOOK_FILENAME} not found in CWD. "
          "Start Jupyter from the notebook folder or update NOTEBOOK_FILENAME.")

print("Saved run to outputs:", OUTPUTS_DIR.resolve())
print("Snapshots in:", SNAPSHOTS_DIR.resolve())

